# M3 — 01 Bronze to Silver (manual)

Set `run_date` to the **ADF ingest folder date** (`YYYY-MM-DD`). Run all **only after checking the settings below**.

- Full refresh: customers, products, supplier price **history**, marketing, web events/quarantine.
- Incremental: orders, payments, order_items for `run_date`.
- DQ metrics are staged for Notebook 02; the final DQ log is not updated here.
- **No M4 business-level deduplication or scheduled automation.**

**Safety:** By default this notebook runs in PREVIEW mode. Review counts first. Set `ALLOW_WRITES=True` only when ready. Each Silver table is a separate Delta commit, so a failed run is not automatically rolled back across all tables.


In [0]:
# Create run_date widget
dbutils.widgets.text("run_date", "2026-11-01", "Run Date (YYYY-MM-DD)")

# Get selected run date
run_date = dbutils.widgets.get("run_date")

print(f"Pipeline Run Date: {run_date}")

In [0]:
from datetime import date
from decimal import Decimal
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable

BRONZE = "abfss://lakehouse@cdstlakehouse.dfs.core.windows.net/bronze"
SILVER = "atliq.silver"
STAGE_TABLE = f"{SILVER}.dq_stage"

# True = before writing, check the guide's seed-data totals (safe while you test).
# Set to False for normal nightly runs (milestone 5), because real data changes the totals.
SEED_CHECKS = False
ALLOW_WRITES = True  # Change to True only after reviewing the preview and existing Silver versions

dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date").strip()
assert run_date, "Type a date (YYYY-MM-DD) in the run_date box at the top, then Run all again"
date.fromisoformat(run_date)
print("Processing date:", run_date)

PLAN = {}     # table name -> cleaned DataFrame. Written only after every check has passed
COUNTS = {}   # (table, rule) -> rows affected. Saved at the end for notebook 02

def count_rule(table_name, rule, n):
    COUNTS[(table_name, rule)] = int(n)

def clean_key(col_name):
    return F.lower(F.regexp_replace(F.trim(F.col(col_name)), r"\s+", " "))

def letters_key(col_name):
    return F.regexp_replace(F.lower(F.trim(F.col(col_name))), r"[^a-z]", "")

def make_map(d):
    return F.create_map(*[F.lit(x) for kv in d.items() for x in kv])

def batch_exists(table):
    """Quiet-night guard (guide 7.4): did ADF land a Parquet file for this date?"""
    path = f"{BRONZE}/{table}/ingest_date={run_date}"
    try:
        return any(f.name.endswith(".parquet") for f in dbutils.fs.ls(path))
    except Exception as e:
        if "FileNotFound" in str(e) or "PATH_NOT_FOUND" in str(e):
            return False
        raise

# ---- cleaning rules, all in one place ----
STATUS_MAP = {"placed": "Placed", "shipped": "Shipped", "delivered": "Delivered",
              "cancelled": "Cancelled", "canceled": "Cancelled", "returned": "Returned"}

PAY_MAP = {"upi": "UPI", "credit card": "Credit Card", "creditcard": "Credit Card",
           "debit card": "Debit Card", "debitcard": "Debit Card",
           "net banking": "Net Banking", "netbanking": "Net Banking",
           "wallet": "Wallet", "cod": "COD", "cash on delivery": "COD"}

CATEGORY_MAP = {"electronics": "Electronics", "home and kitchen": "Home & Kitchen",
                "fashion": "Fashion", "books": "Books", "beauty": "Beauty", "sports": "Sports"}

CHANNEL_MAP = {"metaads": "Meta Ads", "facebookads": "Meta Ads",   # Facebook to Meta: assumption, not in the guide
               "googleads": "Google Ads", "email": "Email", "seo": "SEO",
               "influencer": "Influencer", "influencermarketing": "Influencer"}

CITY_MAP = {"Bangalore": "Bengaluru", "Bombay": "Mumbai", "Calcutta": "Kolkata", "Madras": "Chennai"}

# Delta version of each Silver table before this run (restore points)
for t in ["customers", "products", "supplier_price_list", "marketing_spend",
          "web_events", "orders", "order_items", "payments"]:
    v = DeltaTable.forName(spark, f"{SILVER}.{t}").history(1).select("version").first()[0]
    print(f"version before run: {t:22s} {v}")

## A. Small tables (rebuilt from Bronze every run)

In [0]:
# --- customers ---
cust_cols = spark.table(f"{SILVER}.customers").columns

raw_c = spark.read.parquet(f"{BRONZE}/customers")
n_null_id = raw_c.filter(F.col("customer_id").isNull()).count()

c = (raw_c
     .withColumn("city_raw", F.col("city"))
     .withColumn("email_raw", F.col("email"))
     .withColumn("city", F.initcap(F.regexp_replace(F.trim(F.col("city")), r"\s+", " ")))
     .withColumn("email", F.lower(F.trim(F.col("email"))))
     .withColumn("signup_date", F.to_date("signup_date"))
     .filter(F.col("customer_id").isNotNull())
     .dropDuplicates(["customer_id"]))

old_names = list(CITY_MAP.keys())
n_old_city  = c.filter(F.col("city").isin(*old_names)).count()
n_city_fix  = c.filter(~F.col("city").isin(*old_names) & ~F.col("city").eqNullSafe(F.col("city_raw"))).count()
n_email_fix = c.filter(~F.col("email").eqNullSafe(F.col("email_raw"))).count()

city_expr = F.col("city")
for old, new in CITY_MAP.items():
    city_expr = F.when(F.col("city") == old, new).otherwise(city_expr)

c = (c.withColumn("city", city_expr)
      .withColumn("is_test", F.coalesce(F.col("email").endswith("@atliq-test.com"), F.lit(False))))

n_rows = c.count()
n_test = c.filter(F.col("is_test")).count()
if SEED_CHECKS:
    assert n_rows == 1008 and n_test == 8, f"customers: expected 1008 rows / 8 test, got {n_rows} / {n_test}. NOT written"

PLAN["customers"] = c.select(*cust_cols)
print("customers rows:", n_rows, "| test accounts:", n_test)
count_rule("customers", "null_customer_id", n_null_id)
count_rule("customers", "city_trim_or_case_normalized", n_city_fix)
count_rule("customers", "old_city_names_mapped", n_old_city)
count_rule("customers", "email_lowercase_trim", n_email_fix)
count_rule("customers", "test_accounts_flagged_is_test", n_test)

In [0]:
# --- products: map to the guide's 6 categories ---
prod_cols = spark.table(f"{SILVER}.products").columns
cat_key = F.lower(F.regexp_replace(F.trim(F.regexp_replace(F.col("category"), "&", " and ")), r"\s+", " "))

p = (spark.read.parquet(f"{BRONZE}/products")
     .withColumn("category_raw", F.col("category"))
     .withColumn("name_raw", F.col("product_name"))
     .withColumn("category", F.try_element_at(make_map(CATEGORY_MAP), cat_key))
     .withColumn("product_name", F.regexp_replace(F.trim(F.col("product_name")), r"\s+", " "))
     .filter(F.col("product_id").isNotNull())
     .dropDuplicates(["product_id"]))

assert p.filter(F.col("category").isNull()).count() == 0, "Unknown product category: add it to CATEGORY_MAP. NOT written"
n_prod = p.count()
n_categories = p.select("category").distinct().count()
n_cat = p.filter(~F.col("category").eqNullSafe(F.col("category_raw"))).count()
n_name = p.filter(~F.col("product_name").eqNullSafe(F.col("name_raw"))).count()
if SEED_CHECKS:
    assert n_prod == 60 and n_categories == 6, f"products: expected 60 / 6, got {n_prod} / {n_categories}. NOT written"

PLAN["products"] = p.select(*prod_cols)
print("products rows:", n_prod, "| categories:", n_categories)
count_rule("products", "category_text_normalized_total", n_cat)
count_rule("products", "product_name_text_normalized", n_name)

In [0]:
# --- supplier_price_list: retain cleaned effective-date HISTORY in Silver (M4 selects latest per product) ---
sup_raw = spark.read.parquet(f"{BRONZE}/supplier_price_list")
assert len(sup_raw.columns) == 5, f"Unexpected supplier Bronze columns: {sup_raw.columns}"
sup_raw = sup_raw.toDF("product_id", "product_name", "supplier_name", "supplier_cost", "effective_date")
body = sup_raw.filter(F.lower(F.trim(F.col("product_id"))) != "product_id")
n_header = sup_raw.count() - body.count()
n_date_fmt = body.filter(~F.col("effective_date").rlike(r"^\d{4}-\d{2}-\d{2}$")).count()
n_cost_fmt = body.filter(F.col("supplier_cost").cast("string").rlike(r"(?i)(₹|rs\.?|,)")).count()
n_repeats = body.count() - body.dropDuplicates().count()
body = body.dropDuplicates()
amount = F.regexp_replace(F.regexp_extract("supplier_cost", r"-?[0-9][0-9,]*(\.[0-9]+)?", 0), ",", "")
parsed = (body
    .withColumn("product_id", F.expr("try_cast(trim(product_id) AS INT)"))
    .withColumn("product_name", F.regexp_replace(F.trim("product_name"), r"\s+", " "))
    .withColumn("supplier_name", F.regexp_replace(F.trim("supplier_name"), r"\s+", " "))
    .withColumn("supplier_cost", F.when(amount != "", amount.cast("decimal(12,2)")))
    .withColumn("effective_date", F.coalesce(
        F.to_date(F.try_to_timestamp(F.col("effective_date"), F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(F.col("effective_date"), F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(F.col("effective_date"), F.lit("dd-MMM-yyyy"))))))
known = p.select("product_id").distinct()
history = parsed.join(known, "product_id", "left_semi")
n_unknown = parsed.count() - history.count()
assert history.filter(F.col("supplier_cost").isNull() | F.col("effective_date").isNull() |
                      F.col("supplier_name").isNull()).count() == 0, "Unparsed supplier values: NOT written"
# Check the CURRENT-price checkpoint without discarding historical records.
w = Window.partitionBy("product_id").orderBy(F.col("effective_date").desc())
latest_date = history.withColumn("latest_date", F.max("effective_date").over(Window.partitionBy("product_id")))
conflicts = (latest_date.filter(F.col("effective_date") == F.col("latest_date"))
             .groupBy("product_id").agg(F.countDistinct("supplier_cost", "supplier_name").alias("variants"))
             .filter("variants > 1").count())
assert conflicts == 0, "Conflicting supplier prices on same latest effective date; resolve before write"
current = (history.withColumn("rn", F.row_number().over(
    Window.partitionBy("product_id").orderBy(F.col("effective_date").desc(),
        F.col("supplier_name").asc(), F.col("supplier_cost").asc())))
    .filter("rn = 1").drop("rn"))
n_cur = current.count()
cur_total = current.agg(F.sum("supplier_cost")).first()[0]
if SEED_CHECKS:
    assert n_cur == 60 and cur_total == Decimal("56871.53"),         f"supplier current-price checkpoint expected 60 / 56871.53; got {n_cur} / {cur_total}"
PLAN["supplier_price_list"] = history.select(*spark.table(f"{SILVER}.supplier_price_list").columns)
print("supplier HISTORY rows:", history.count(), "| CURRENT prices:", n_cur, "| current total:", cur_total)
count_rule("supplier_price_list", "header_row_removed", n_header)
count_rule("supplier_price_list", "exact_duplicate_rows_removed", n_repeats)
count_rule("supplier_price_list", "unknown_product_rows_dropped", n_unknown)
count_rule("supplier_price_list", "supplier_date_format_parsed", n_date_fmt)
count_rule("supplier_price_list", "supplier_cost_currency_parsed", n_cost_fmt)


In [0]:
# --- marketing_spend: remove repeats BEFORE mapping channels (mapping first would create false repeats) ---
mkt_cols = spark.table(f"{SILVER}.marketing_spend").columns

mkt_raw = spark.read.parquet(f"{BRONZE}/marketing_spend")
n_clicks_comma = mkt_raw.dropDuplicates().filter(F.col("clicks").cast("string").contains(",")).count()

typed = (mkt_raw
         .withColumn("spend_date", F.try_to_date("spend_date", "yyyy-MM-dd"))
         .withColumn("spend_amount", F.regexp_replace(F.col("spend_amount").cast("string"), ",", "").cast("decimal(12,2)"))
         .withColumn("clicks", F.regexp_replace(F.col("clicks"), ",", "").cast("int")))

deduped   = typed.dropDuplicates()
n_repeats = typed.count() - deduped.count()
complete  = deduped.dropna()
n_blank   = deduped.count() - complete.count()

n_fb = complete.filter(letters_key("channel") == "facebookads").count()
cleaned = (complete
           .withColumn("channel_raw", F.col("channel"))
           .withColumn("channel", F.try_element_at(make_map(CHANNEL_MAP), letters_key("channel"))))

n_unmapped = cleaned.filter(F.col("channel").isNull()).count()
assert n_unmapped == 0, "Unmapped channel spelling (add it to CHANNEL_MAP): NOT written"
n_text = cleaned.filter((F.col("channel") != F.col("channel_raw")) &
                        (letters_key("channel_raw") != "facebookads")).count()
n_negative = cleaned.filter(F.col("spend_amount") < 0).count()

tot = cleaned.agg(F.count("*"), F.sum("spend_amount"), F.sum("clicks")).first()
if SEED_CHECKS:
    assert tot[0] == 3037 and tot[1] == Decimal("8920850.16") and tot[2] == 3274996, \
        f"marketing: expected 3037 / 8920850.16 / 3274996, got {tot[0]} / {tot[1]} / {tot[2]}. NOT written"

PLAN["marketing_spend"] = cleaned.select(*mkt_cols)
print("marketing rows:", tot[0], "| spend:", tot[1], "| clicks:", tot[2])
count_rule("marketing_spend", "exact_repeats_dropped", n_repeats)
count_rule("marketing_spend", "blank_spend_rows_dropped", n_blank)
count_rule("marketing_spend", "channel_text_normalized", n_text)
count_rule("marketing_spend", "facebook_ads_mapped_to_meta_ads", n_fb)
count_rule("marketing_spend", "unmapped_channel_to_null", n_unmapped)
count_rule("marketing_spend", "marketing_clicks_comma_parsed", n_clicks_comma)
count_rule("marketing_spend", "negative_spend_rows_retained", n_negative)

In [0]:
# --- clickstream -> web_events and quarantine. Guide order: one row per event, session, bots, products ---
event_schema = StructType([
    StructField("event_id", StringType()), StructField("session_id", StringType()),
    StructField("customer_id", IntegerType()), StructField("event_type", StringType()),
    StructField("event_ts", StringType()), StructField("product_id", IntegerType()),
    StructField("order_id", IntegerType()), StructField("device", StringType()),
    StructField("user_agent", StringType()), StructField("utm_source", StringType()),
])
EVENT_COLS = ["event_id", "session_id", "customer_id", "event_type", "event_ts",
              "product_id", "order_id", "device", "user_agent", "utm_source"]
BOT_RE = r"(?i)(bot|crawler|spider|python-requests)"
EPOCH_RE = r"^\d+$"
ISO_FMT = "yyyy-MM-dd'T'HH:mm:ss.SSSX"

parsed_lines = spark.read.text(f"{BRONZE}/clickstream").withColumn("e", F.from_json("value", event_schema))

quarantine = (
    parsed_lines
    .filter(F.col("e.event_id").isNull())
    .select("value")
    .distinct()
)
n_quarantined = quarantine.count()
print("Total quarantine rows:", quarantine.count())
print("Distinct quarantine rows:", quarantine.distinct().count())

ev = (parsed_lines.filter(F.col("e.event_id").isNotNull()).select("e.*")
      .withColumn("event_ts_raw", F.col("event_ts"))
      .withColumn("event_ts", F.when(F.col("event_ts").rlike(EPOCH_RE),
                                     F.timestamp_millis(F.col("event_ts").cast("long")))
                               .otherwise(F.try_to_timestamp("event_ts", F.lit(ISO_FMT)))))
n_valid = ev.count()

# 1. one row per event_id (the latest timestamp wins, so the choice is repeatable)
w = Window.partitionBy("event_id").orderBy(F.col("event_ts").desc_nulls_last())
ev = ev.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
n_dups = n_valid - ev.count()

# 2. drop events without a session
n_before = ev.count()
ev = ev.filter(F.col("session_id").isNotNull() & (F.trim(F.col("session_id")) != ""))
n_nosession = n_before - ev.count()

# 3. drop whole bot sessions
bot_sessions = (ev.filter(F.coalesce(F.col("user_agent"), F.lit("")).rlike(BOT_RE))
                  .select("session_id").distinct())
n_bot_sessions = bot_sessions.count()
n_before = ev.count()
ev = ev.join(bot_sessions, "session_id", "left_anti")
n_bots = n_before - ev.count()

# 4. drop events for unknown products (events without a product are kept)
known_p = p.select(F.col("product_id").alias("_pid"))
n_before = ev.count()
ev = (ev.join(known_p, F.col("product_id") == F.col("_pid"), "left")
        .filter(F.col("product_id").isNull() | F.col("_pid").isNotNull()).drop("_pid"))
n_unknown_prod = n_before - ev.count()

# 5. normalise event_type
n_type_fix = ev.filter(~F.col("event_type").eqNullSafe(F.lower(F.trim(F.col("event_type"))))).count()
n_epoch = ev.filter(F.col("event_ts_raw").rlike(EPOCH_RE)).count()
n_bad_ts = ev.filter(F.col("event_ts").isNull()).count()
ev = ev.withColumn("event_type", F.lower(F.trim(F.col("event_type"))))

assert n_bad_ts == 0, "Unparseable clickstream event timestamp: NOT written"
n_events = ev.count()
n_sessions = ev.select("session_id").distinct().count()
if SEED_CHECKS:
    assert n_quarantined == 426 and n_events == 93436 and n_sessions == 23193 and n_bad_ts == 0, \
        f"clickstream: expected 426 / 93436 / 23193 / 0 bad timestamps, got {n_quarantined} / {n_events} / {n_sessions} / {n_bad_ts}. NOT written"

PLAN["web_events_quarantine"] = quarantine
PLAN["web_events"] = ev.select(*EVENT_COLS)
print("quarantined:", n_quarantined, "| web_events:", n_events, "| sessions:", n_sessions, "| unparsed timestamps:", n_bad_ts)
count_rule("web_events", "malformed_json_quarantined", n_quarantined)
count_rule("web_events", "duplicate_event_ids_removed", n_dups)
count_rule("web_events", "missing_session_ids_dropped", n_nosession)
count_rule("web_events", "bot_sessions_identified", n_bot_sessions)
count_rule("web_events", "bot_session_events_dropped", n_bots)
count_rule("web_events", "unknown_product_events_dropped", n_unknown_prod)
count_rule("web_events", "event_type_normalized", n_type_fix)
count_rule("web_events", "event_ts_epoch_ms_parsed", n_epoch)
count_rule("web_events", "event_ts_unparseable", n_bad_ts)

## A-write. Write the small tables (only reached if every check above passed)

In [0]:
print("PREVIEW: full-refresh rows to write:")

for name, df in PLAN.items():
    print(name, df.count())

if ALLOW_WRITES:
    for name, df in PLAN.items():
        (df.write.format("delta")
           .mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(f"{SILVER}.{name}"))

        print("written:", f"{SILVER}.{name}")
else:
    print("PREVIEW ONLY: No Silver tables were modified.")

## B. Orders, payments and order_items (Delta MERGE, today's batch only)

In [0]:
present = {t: batch_exists(t) for t in ["orders", "order_items", "payments"]}
print("batch present for", run_date, ":", present)

In [0]:

# --- orders ---
if present["orders"]:
    raw_o = spark.read.parquet(f"{BRONZE}/orders/ingest_date={run_date}")
    n_raw = raw_o.count()

    latest = (
        raw_o.filter(F.col("order_id").isNotNull())
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("order_id")
                .orderBy(F.col("updated_at").desc_nulls_last())
            )
        )
        .filter("rn = 1")
        .drop("rn", "ingest_date")
    )

    n_collapsed = n_raw - latest.count()

    n_legacy = latest.filter(
        F.to_date("order_date") == F.lit("1900-01-01").cast("date")
    ).count()

    orders_batch = (
        latest
        .withColumn("status_raw", F.col("status"))
        .withColumn(
            "status",
            F.try_element_at(make_map(STATUS_MAP), clean_key("status"))
        )
        .withColumn(
            "order_date",
            F.when(
                F.to_date("order_date") == F.lit("1900-01-01").cast("date"),
                F.to_date("created_at")
            ).otherwise(F.to_date("order_date"))
        )
    )

    assert orders_batch.filter(F.col("status").isNull()).count() == 0, \
        "Unknown order status: add it to STATUS_MAP"

    n_status = orders_batch.filter(
        F.col("status") != F.col("status_raw")
    ).count()

    orders_batch = orders_batch.drop("status_raw")

    print("orders in batch:", orders_batch.count())

    if ALLOW_WRITES:
        (DeltaTable.createIfNotExists(spark)
            .tableName(f"{SILVER}.orders")
            .addColumns(orders_batch.schema)
            .execute())

        (DeltaTable.forName(spark, f"{SILVER}.orders").alias("t")
            .merge(orders_batch.alias("s"), "t.order_id = s.order_id")
            .whenMatchedUpdate(
                condition="s.updated_at > t.updated_at",
                set={
                    "customer_id": "s.customer_id",
                    "order_date": "s.order_date",
                    "status": "s.status",
                    "order_amount": "s.order_amount",
                    "created_at": "s.created_at",
                    "updated_at": "s.updated_at"
                }
            )
            .whenNotMatchedInsert(values={
                "order_id": "s.order_id",
                "customer_id": "s.customer_id",
                "order_date": "s.order_date",
                "status": "s.status",
                "order_amount": "s.order_amount",
                "created_at": "s.created_at",
                "updated_at": "s.updated_at",
                "ingest_date": f"'{run_date}'"
            })
            .execute())

        print("Orders MERGE completed.")

    else:
        print("PREVIEW ONLY: Orders MERGE skipped.")

    count_rule("orders", "status_text_mapped", n_status)
    count_rule("orders", "legacy_order_date_fixed_in_batch", n_legacy)
    count_rule("orders", "order_versions_collapsed_in_batch", n_collapsed)

else:
    print("orders: no batch for", run_date, "- skipped")


In [0]:

# --- payments ---
if present["payments"]:
    raw_p = spark.read.parquet(f"{BRONZE}/payments/ingest_date={run_date}")
    n_raw = raw_p.count()

    latest = (
        raw_p.filter(F.col("payment_id").isNotNull())
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("payment_id")
                .orderBy(F.col("updated_at").desc_nulls_last())
            )
        )
        .filter("rn = 1")
        .drop("rn", "ingest_date")
    )

    n_collapsed = n_raw - latest.count()

    n_retries = (
        latest.count()
        - latest.dropDuplicates(["order_id"]).count()
    )

    payments_batch = (
        latest
        .withColumn("method_raw", F.col("method"))
        .withColumn(
            "method",
            F.try_element_at(make_map(PAY_MAP), clean_key("method"))
        )
    )

    assert payments_batch.filter(F.col("method").isNull()).count() == 0, \
        "Unknown payment method: add it to PAY_MAP"

    n_method = payments_batch.filter(
        ~F.col("method").eqNullSafe(F.col("method_raw"))
    ).count()

    payments_batch = payments_batch.drop("method_raw")

    print("payments in batch:", payments_batch.count())

    if ALLOW_WRITES:
        (DeltaTable.createIfNotExists(spark)
            .tableName(f"{SILVER}.payments")
            .addColumns(payments_batch.schema)
            .execute())

        (DeltaTable.forName(spark, f"{SILVER}.payments").alias("t")
            .merge(payments_batch.alias("s"), "t.payment_id = s.payment_id")
           .whenMatchedUpdate(
    condition="s.updated_at > t.updated_at",
    set={
        "order_id": "s.order_id",
        "amount": "s.amount",
        "method": "s.method",
        "paid_at": "s.paid_at",
        "updated_at": "s.updated_at"
    }
)
.whenNotMatchedInsert(values={
    "payment_id": "s.payment_id",
    "order_id": "s.order_id",
    "amount": "s.amount",
    "method": "s.method",
    "paid_at": "s.paid_at",
    "updated_at": "s.updated_at",
    "ingest_date": f"'{run_date}'"
})
            .execute())

        print("Payments MERGE completed.")

    else:
        print("PREVIEW ONLY: Payments MERGE skipped.")

    count_rule("payments", "method_text_mapped", n_method)
    count_rule("payments", "payment_versions_collapsed_in_batch", n_collapsed)
    count_rule("payments", "repeated_order_payments_retained_in_batch", n_retries)

else:
    print("payments: no batch for", run_date, "- skipped")


In [0]:

# --- order_items: insert-only ---
if present["order_items"]:
    raw_i = spark.read.parquet(
        f"{BRONZE}/order_items/ingest_date={run_date}"
    )
    n_raw = raw_i.count()

    items_batch = (
        raw_i.filter(F.col("order_item_id").isNotNull())
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("order_item_id")
                .orderBy(F.col("created_at").desc_nulls_last())
            )
        )
        .filter("rn = 1")
        .drop("rn", "ingest_date")
    )

    n_dup_ids = n_raw - items_batch.count()

    n_business_dups = (
        items_batch.count()
        - items_batch.dropDuplicates([
            "order_id", "product_id", "quantity",
            "item_price", "created_at"
        ]).count()
    )

    n_invalid = items_batch.filter(
        F.expr("try_cast(quantity AS INT)").isNull()
        | (F.expr("try_cast(quantity AS INT)") <= 0)
        | F.expr("try_cast(item_price AS DECIMAL(12,2))").isNull()
        | (F.expr("try_cast(item_price AS DECIMAL(12,2))") <= 0)
    ).count()

    if n_invalid:
        print(
            "WARNING: order items with zero, negative or missing quantity/price:",
            n_invalid
        )

    print("items in batch:", items_batch.count())

    if ALLOW_WRITES:
        (DeltaTable.createIfNotExists(spark)
            .tableName(f"{SILVER}.order_items")
            .addColumns(items_batch.schema)
            .execute())

        (DeltaTable.forName(spark, f"{SILVER}.order_items").alias("t")
            .merge(items_batch.alias("s"),
                   "t.order_item_id = s.order_item_id")
            .whenNotMatchedInsert(values={
                "order_item_id": "s.order_item_id",
                "order_id": "s.order_id",
                "product_id": "s.product_id",
                "quantity": "s.quantity",
                "item_price": "s.item_price",
                "created_at": "s.created_at",
                "ingest_date": f"'{run_date}'"
            })
            .execute())

        print("Order Items MERGE completed.")

    else:
        print("PREVIEW ONLY: Order Items MERGE skipped.")

    count_rule(
        "order_items", "duplicate_item_ids_removed_in_batch", n_dup_ids
    )
    count_rule(
        "order_items", "business_duplicate_lines_retained_in_batch",
        n_business_dups
    )
    count_rule(
        "order_items", "invalid_quantity_or_price_found", n_invalid
    )

else:
    print("order_items: no batch for", run_date, "- skipped")


## C. Save the counts for notebook 02, then final checks

In [0]:

if ALLOW_WRITES:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STAGE_TABLE}
        (run_date STRING, table_name STRING,
         rule STRING, rows_affected LONG)
    """)

    rows = [
        (run_date, t, r, n)
        for (t, r), n in COUNTS.items()
    ]

    new = spark.createDataFrame(
        rows,
        "run_date STRING, table_name STRING, rule STRING, rows_affected LONG"
    )

    (DeltaTable.forName(spark, STAGE_TABLE).alias("t")
        .merge(
            new.alias("s"),
            """t.run_date = s.run_date
               AND t.table_name = s.table_name
               AND t.rule = s.rule"""
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    print("saved", len(rows), "counts for", run_date, "in", STAGE_TABLE)

else:
    print("PREVIEW ONLY: DQ staging skipped.")


In [0]:
for t in ["customers", "products", "supplier_price_list", "marketing_spend", "web_events",
          "web_events_quarantine", "orders", "order_items", "payments"]:
    print(f"{SILVER}.{t:24s}", spark.table(f"{SILVER}.{t}").count())

print("\npayment methods:", sorted(r[0] for r in spark.table(f"{SILVER}.payments").select("method").distinct().collect()))
print("order statuses: ", sorted(r[0] for r in spark.table(f"{SILVER}.orders").select("status").distinct().collect()))

new_ids = (spark.table(f"{SILVER}.orders")
           .filter(F.col("order_date") > F.lit("2026-08-31").cast("date")).select("order_id"))
print("\norders after the seed period:", new_ids.count())
print("  with items:   ", spark.table(f"{SILVER}.order_items").join(new_ids, "order_id").select("order_id").distinct().count())
print("  with payments:", spark.table(f"{SILVER}.payments").join(new_ids, "order_id").select("order_id").distinct().count())